# Lab 3: Pipeline Health Check — Finding Data Corruption Through EDA

**ECON 5200: Causal Machine Learning & Applied Analytics**

This lab uses a **diagnosis-first** format. You will:
1. Find and fix 5 planted data quality issues in a deliberately corrupted dataset using EDA alone
2. Run distribution diagnostics that detect shifts between training and inference data, and judge how far to trust them
3. Compare manual EDA to automated profiling (ydata-profiling) and identify what each misses
4. Read and run a finished `eda_utils.py` module that collects these checks into functions you can import

**Time estimate:** about 70 minutes (Part 0: 15 min, Part 1: 20 min, Part 2: 15 min, Part 3: 10 min, Part 4: 10 min)

**Prerequisites:** Chapter 3 reading, including [5200-DEPTH] sections on distribution diagnostics and data pipeline monitoring

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only where this lab says so.

> ### What you write, and what you run
>
> **You write:**
> - **Part 1, the fix cell.** It starts with `df_clean = df.copy()` and five empty slots. You write about a dozen lines until the verification cell prints six PASSes. The diagnosis of the five bugs goes in the text cell above it.
> - **Parts 2 and 3:** four interpretation answers, one of them a table, in text cells.
> - **The AI Expansion:** the AI writes that code; you write the three answers about what it got wrong.
>
> **You run and read:** everything else. Part 0, Part 2's PSI code and Part 4's `eda_utils.py` are working code to learn from, not to retype.
>
> **Reading this for Problem Set 1?** Part 4's `eda_utils.py` is the closest model you have for `audit_report(df)`: short functions with one job each, a docstring and type hints on every one, and a loop over the columns that computes one result per column (`eda_summary`'s outlier count). The difference: `audit_report` returns a DataFrame with one row per column, where `eda_summary` returns a dict.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

<!-- 5200-onramp -->
## Part 0: Build It First (GUIDED — 15 min)

Part 1 hands you a merged dataset with five things wrong with it and asks you
to find them all using EDA alone. Before you can spot corruption, you need to
know what a healthy profile of a clean panel looks like, and you need the
pandas verbs that produce one.

**This lab's slice of Python:** `.groupby()`, `.map()` / `.apply()`, the
three loop helpers you will read constantly (`zip`, `enumerate`, `range`), and
`try` / `except`.

### Step 0a: groupby, map, apply

`.groupby(col)` **splits** the table into one group per distinct value,
**applies** an aggregation to each group, and **combines** the answers: split,
apply, combine. What comes back has the grouping column as its **index**;
`.reset_index()` turns it back into an ordinary column. `.agg([...])` asks for
several statistics at once.

`.map()` applies something to every value of one column: a function, or a
dict that recodes labels. `lambda x: x * 1000` is a one-line function with no
name, written in place so `.map()` can run it once per value.
`.apply(..., axis=1)` hands your function a whole **row** instead of one value.

In [ ]:
# GIVEN — run as-is
# Step 0a: groupby, map, apply

clean = pd.DataFrame({
    "country":   ["USA", "USA", "China", "China", "Brazil", "Brazil"],
    "year":      [2022, 2023, 2022, 2023, 2022, 2023],
    "gdp":       [25.7, 27.4, 17.9, 18.5, 1.92, 2.17],     # trillions USD
    "region":    ["Americas", "Americas", "Asia", "Asia", "Americas", "Americas"],
})

print("Mean GDP by region:")
print(clean.groupby("region")["gdp"].mean().round(2))

print("\nSeveral statistics at once — .agg():")
print(clean.groupby("country")["gdp"].agg(["count", "mean", "max"]).round(2))

clean["gdp_billions"] = clean["gdp"].map(lambda x: x * 1000)
clean["region_code"] = clean["region"].map({"Americas": "AM", "Asia": "AS"})

# {value:.2f} inside an f-string prints the value with 2 decimals
clean["label"] = clean.apply(
    lambda row: f"{row['country']} {row['year']}: ${row['gdp']:.2f}T", axis=1
)
# "\n".join(...) glues the labels into one string, one per line
print("\n" + "\n".join(clean["label"]))

### Step 0b: the loop helpers, and try / except

- `zip()` walks two lists side by side, one pair per step.
- `enumerate()` walks a list while counting positions (0, 1, 2 ...). This is
  the form you will read in every multi-panel plotting cell.
- `range(n)` repeats n times. Naming the loop variable `_` means "I do not need
  this value".

`try` / `except` catches **one named** failure and carries on. Always name the
exception you expect: a bare `except:` also swallows real bugs and leaves you
debugging blind.

The first loop uses two format specs: `:<8` pads to 8 characters, left-aligned,
and `:>6.2f` right-aligns in 6 characters with 2 decimals.

In [ ]:
# GIVEN — run as-is
# Step 0b: the loop helpers, and try / except

countries = ["USA", "China", "Brazil"]
gdp       = [27.4, 18.5, 2.17]

for name, value in zip(countries, gdp):
    print(f"  {name:<8}${value:>6.2f}T")

print()
for i, name in enumerate(countries):
    print(f"  panel {i}: {name}")

total = 0
for _ in range(3):
    total += 1
print(f"\nRepeated 3 times, total = {total}")

raw = ["3.1", "2.5", "n/a", "4.5", ""]
cleaned = []
for item in raw:
    try:
        cleaned.append(float(item))
    except ValueError:              # float("n/a") and float("") raise ValueError
        cleaned.append(np.nan)      # np.nan is "missing number"
print(f"\nRaw:     {raw}")
print(f"Cleaned: {cleaned}")

### Step 0c: what a healthy profile looks like

Part 1 hands you a corrupted version of a panel shaped exactly like this one.
Run the same four checks on clean data first, so you know what "fine" reads
like. These four are the whole EDA checklist:

| Check | Question |
|---|---|
| **STRUCTURE** | shape, dtypes: one row per what? |
| **DISTRIBUTIONS** | `describe()`: any impossible values? |
| **COMPLETENESS** | nulls, and duplicate keys |
| **RANGES** | does every column obey its domain constraint? |

Two idioms in the RANGES check: `|` means "or", element by element, and
`.sum()` of a True/False column counts the Trues.

In [ ]:
# GIVEN — run as-is
# Step 0c: the four checks on a healthy panel

rng = np.random.default_rng(42)
n_countries = 10
years = list(range(2000, 2024))

# one dict per row; pd.DataFrame turns the list of dicts into a table
healthy_rows = []
for i in range(n_countries):
    for y in years:
        healthy_rows.append({"country": f"C{i}", "year": y,
                             "gdp": rng.uniform(500, 25000),
                             "population": rng.uniform(5, 1400),
                             "life_expectancy": rng.uniform(55, 85),
                             "trade_pct": rng.uniform(20, 85)})
healthy = pd.DataFrame(healthy_rows)

print("1. STRUCTURE")
print(f"   shape {healthy.shape} — expect {n_countries} countries x {len(years)} years "
      f"= {n_countries * len(years)}")
n_dup = healthy.duplicated(subset=["country", "year"]).sum()
print(f"   one row per country-year? duplicates: {n_dup}")

print("\n2. DISTRIBUTIONS")
print(healthy[["gdp", "population", "life_expectancy", "trade_pct"]]
      .describe().loc[["min", "50%", "max"]].round(1).to_string())

print("\n3. COMPLETENESS")
# .isna().sum() counts nulls per column; the second .sum() adds up the columns
print(f"   nulls, all columns together: {healthy.isna().sum().sum()}")

print("\n4. RANGES — every column against its domain constraint")
constraints = {"gdp": (0, 1e6), "population": (0, 2000),
               "life_expectancy": (0, 120), "trade_pct": (0, 200)}
for col, limits in constraints.items():
    lo, hi = limits
    bad = ((healthy[col] < lo) | (healthy[col] > hi)).sum()
    print(f"   {col:<18} outside [{lo}, {hi}]: {bad}")

### Now diagnose

Every number above is what healthy looks like: no duplicate keys, no nulls, no
value outside its domain, and a `describe()` whose min and max are plausible
for what the column claims to measure.

Part 1's dataset fails several of these. Run the same four checks on it and let
the *contrast* tell you where to look — that is faster and far more reliable
than reading the generating code, and it is what you would actually do with a
pipeline you did not write.

---

## Part 1: The Corrupted Dataset (DIAGNOSTIC — 20 min)

The cell below builds a country panel merged from three sources. Five things
went wrong in the merge. Using only the EDA checks from Part 0 and Chapter 3,
find all five, explain what is wrong with each, and fix it, before fitting any
model.

In [ ]:
# GIVEN — run as-is. This builds the corrupted dataset; do not modify it.
np.random.seed(42)

countries = ["USA", "China", "India", "Brazil", "Germany",
             "Japan", "UK", "France", "Canada", "Australia"]
years = list(range(2000, 2024))

# the clean panel
rows = []
for c in countries:
    for y in years:
        gdp = np.random.uniform(500, 25000)  # GDP in millions USD
        pop = np.random.uniform(5, 1400)      # population in millions
        le = np.random.uniform(55, 85)         # life expectancy in years
        trade = np.random.uniform(20, 85)      # trade as % of GDP
        rows.append([c, y, gdp, pop, le, trade])

df = pd.DataFrame(rows, columns=["country", "year", "gdp", "population",
                                  "life_expectancy", "trade_pct"])

# corruption 1
neg_idx = np.random.choice(df.index, size=10, replace=False)
df.loc[neg_idx, "gdp"] = df.loc[neg_idx, "gdp"] * -1

# corruption 2
month_idx = np.random.choice(df.index, size=15, replace=False)
df.loc[month_idx, "life_expectancy"] = df.loc[month_idx, "life_expectancy"] * 12

# corruption 3
dup_idx = np.random.choice(df.index, size=20, replace=False)
dup_rows = df.loc[dup_idx].copy()
dup_rows["gdp"] = dup_rows["gdp"] * np.random.uniform(0.98, 1.02, size=20)
df = pd.concat([df, dup_rows], ignore_index=True)

# corruption 4
decimal_idx = np.random.choice(df.index, size=30, replace=False)
df.loc[decimal_idx, "trade_pct"] = df.loc[decimal_idx, "trade_pct"] / 100
impossible_idx = np.random.choice(
    df.index.difference(decimal_idx), size=10, replace=False
)
df.loc[impossible_idx, "trade_pct"] = np.random.uniform(200, 500, size=10)

# corruption 5
unit_idx = np.random.choice(df.index, size=50, replace=False)
df.loc[unit_idx, "gdp"] = df.loc[unit_idx, "gdp"] / 1000

# shuffle the rows
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# GIVEN — run as-is. The bugs are visible if you look.

print("=== First 10 rows ===")
display(df.head(10))

print("\n=== Summary statistics ===")
display(df.describe().round(2))

print("\n=== Country-year pairs with more than one row ===")
# .size() counts the rows in each country-year group
dupes = df.groupby(["country", "year"]).size()
print(dupes[dupes > 1].head(10))

### Your Diagnosis

For each of the 5 bugs, write:
1. **What is wrong** (1 sentence identifying the specific issue)
2. **How you found it** (which EDA technique — describe output, histogram, scatter, etc.)
3. **Why it matters** (what downstream analysis would this corrupt?)
4. **Your fix** (show corrected code)

---

**Bug 1** — name it yourself:

*Your answer here:*

**Bug 2** — name it yourself:

*Your answer here:*

**Bug 3** — name it yourself:

*Your answer here:*

**Bug 4** — name it yourself:

*Your answer here:*

**Bug 5** — name it yourself:

*Your answer here:*

### Before you write the fixes

Two of the fixes need a judgment, not just a line of code.

- **Impossible here is not always impossible everywhere.** Trade above 100% of
  GDP is real: Singapore and Luxembourg re-export, so gross trade can exceed
  value added. When you drop or cap a value, take the cutoff from a gap in
  *this* data, and say in your diagnosis whether you dropped or capped, and why.
- **When one column holds two scales, take the threshold from the gap between
  them,** not from a round number like 1.

In [ ]:
# YOUR FIXES — one block per bug, in the order of your diagnosis above

df_clean = df.copy()

# FIX 1


# FIX 2


# FIX 3


# FIX 4


# FIX 5


print(f"Cleaned shape: {df_clean.shape}")
display(df_clean.describe().round(2))

### Verification Checkpoint

After fixing all 5 bugs, `df_clean.describe()` should show:
- **GDP:** min > 0, max in the low thousands to ~25,000 range
- **life_expectancy:** min > 50, max < 100
- **No duplicates:** `df_clean.duplicated(subset=['country', 'year']).sum()` == 0
- **trade_pct:** min > 0, max <= 100
- **GDP units:** min >= 100, so no row is left in thousands

Run the cell below to verify:

In [ ]:
# VERIFICATION — run after your fixes

checks = {
    "GDP all positive": df_clean["gdp"].min() > 0,
    "Life expectancy max < 100": df_clean["life_expectancy"].max() < 100,
    "No duplicate country-years": df_clean.duplicated(
        subset=["country", "year"]
    ).sum() == 0,
    "Trade pct max <= 100": df_clean["trade_pct"].max() <= 100,
    "Trade pct min > 0": df_clean["trade_pct"].min() > 0,
    "GDP min >= 100": df_clean["gdp"].min() >= 100,
}

for name, passed in checks.items():
    if passed:
        print(f"  [PASS] {name}")
    else:
        print(f"  [FAIL] {name}")

print()
if all(checks.values()):
    print("All six checks passed.")
else:
    print("Some checks failed. Review your fixes above.")

### Why GDP is stored in millions

A GDP column has to hold Tuvalu and the United States in the same dtype: six
orders of magnitude. Choose the unit from the **smallest** value you must
represent, not the largest. In millions the tiniest economies are still
readable whole numbers; in billions they round toward 0.06, and in trillions
they vanish. That is why the UN National Accounts and Eurostat both publish in
millions.

It is also why the unit bug was findable. In millions the clean rows sit at
500–25,000 and the corrupted ones at 0.5–25, a clean factor of 1,000 apart.
Had the column been in trillions, the legitimate values would already be
0.5–25, sitting exactly on top of the corruption, with no gap to find.

## Part 2: Distribution Shift Detection (EXTEND — 15 min)

A clean dataset today doesn't guarantee a clean dataset tomorrow. In production, the data generating process can change — a new data source, a policy shock, a pandemic. This part measures whether the distribution of incoming data has drifted from the training data.

The three cells below are given. The first splits your `df_clean` and plants a
shift; the second draws each column's training and inference distributions on
top of each other; the third puts a number on the gap, the **Population
Stability Index** (PSI). Your work in this part is reading them: which column
has moved, how big a PSI is, and which of the four values you trust. Then answer
the two questions below the checkpoint.

> **Check before you start.** Part 2 runs on your `df_clean`, so its numbers depend on Part 1. With all six Part 1 checks passing you will see 230 rows, a 138 / 92 split, and a GDP PSI of 2.4589. If you see 260 rows, 156 / 104 and a PSI near 1.76, your Part 1 fixes are not applied yet: go back and finish them first.


In [ ]:
# GIVEN — run as-is: split the cleaned data, then shift GDP in the new part

# the rows are already shuffled, so the first 60% is a random training set
split = int(len(df_clean) * 0.6)
train = df_clean.iloc[:split].copy()
inference = df_clean.iloc[split:].copy()

# the planted shift: inference GDP is 30% higher, as if a new source reported it differently
inference["gdp"] = inference["gdp"] * 1.3

print(f"Training set: {len(train)} rows")
print(f"Inference set: {len(inference)} rows")
print(f"\nTraining GDP mean: {train['gdp'].mean():.0f}")
print(f"Inference GDP mean: {inference['gdp'].mean():.0f}")

In [ ]:
# GIVEN — run as-is: which column's two curves have come apart?

numeric_cols = ["gdp", "population", "life_expectancy", "trade_pct"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()      # the 2 x 2 grid of panels as a list of 4

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    train[col].plot.kde(ax=ax, label="Training", color="blue")
    inference[col].plot.kde(ax=ax, label="Inference", color="red", linestyle="--")
    ax.set_title(col)
    ax.legend()

plt.suptitle("Distribution Shift Detection: Training vs. Inference")
plt.tight_layout()
plt.show()

In [ ]:
# GIVEN — run as-is: the Population Stability Index for each column

def compute_psi(expected, actual, n_bins=10):
    """PSI = sum over bins of (actual - expected) * ln(actual / expected), using each bin's share."""
    # the bins span BOTH samples, so no value from either falls off the ends
    breakpoints = np.linspace(
        min(np.min(expected), np.min(actual)),
        max(np.max(expected), np.max(actual)),
        n_bins + 1
    )

    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts = np.histogram(actual, bins=breakpoints)[0]

    # adding a small eps keeps an empty bin from producing log(0)
    eps = 1e-4
    expected_pct = expected_counts / len(expected) + eps
    actual_pct = actual_counts / len(actual) + eps

    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi


print("=== Population Stability Index (PSI) ===")
print(f"{'Column':<20} {'PSI':>8}  {'Interpretation'}")
print("-" * 55)

# 0.1 and 0.25 are the conventional cutoffs; the checkpoint below tests them
for col in numeric_cols:
    psi_val = compute_psi(train[col].values, inference[col].values)
    if psi_val < 0.1:
        interp = "Stable"
    elif psi_val < 0.25:
        interp = "Moderate shift"
    else:
        interp = "SIGNIFICANT SHIFT"
    print(f"{col:<20} {psi_val:>8.4f}  {interp}")

### Verification Checkpoint

Your PSI results should show:
- **GDP:** PSI > 0.25 — the one column we deliberately shifted (multiplied by 1.3)
- **population, life_expectancy, trade_pct:** anywhere from roughly 0.05 to 0.25,
  even though *nothing was shifted in any of them*

That second line is not a typo, and it is the more useful half of this
checkpoint. Measured on the answer key's `df_clean` (230 rows, impossible
trade values dropped) the three untouched columns return
**0.092, 0.164 and 0.234** — two of them above the conventional 0.1
"no shift" line and one nearly at the 0.25 "significant shift" line, with no
drift applied at all.

The reason is sample size. At n = 138 against 92 over 10 bins, PSI already has
a sampling distribution whose mass sits in the low tenths from binning noise
alone. The 0.1 / 0.25 cutoffs come from credit-scoring practice at n in the
tens of thousands and do not transfer to a hundred rows.

**A drift threshold without a null distribution is a superstition.** Before
alarming on PSI in production, simulate the PSI you get from two random splits
of the *same undrifted* data and use that distribution's upper tail as the
alarm level. The GDP column is the only one here whose PSI is large enough to
survive that comparison.

### Interpretation Questions

**Q1:** Why is detecting distribution shift important before deploying a model to production? What happens to predictions if the input distribution changes but the model doesn't know?

*Your answer here:*

**Q2:** A model trained on pre-COVID economic data is deployed in 2021. Which columns in our dataset likely shifted most? How would you design a monitoring system to catch this?

*Your answer here:*

## Part 3: Manual vs. Automated EDA (COMPARE — 10 min)

You just spent 20 minutes manually finding 5 bugs. Could automated profiling have caught them faster? Let's find out.

The cell below profiles the **original, corrupted** `df`. If ydata-profiling is
not installed, it prints four numbers from `df` instead: answer the questions
below from what you *expect* a profiler would flag, given what you know about
the five bugs.

In [ ]:
# GIVEN — run as-is: profile the corrupted df, before your fixes

try:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(
        df,
        title="EDA Report: Corrupted Economic Dataset",
        minimal=True,  # a shorter report that runs faster
    )
    profile.to_notebook_iframe()
except ImportError:
    print("ydata-profiling is not installed. To install: !pip install ydata-profiling")
    print()
    print(f"GDP min:              {df['gdp'].min():.2f}")
    print(f"Life expectancy max:  {df['life_expectancy'].max():.1f}")
    print(f"Duplicate rows:       {df.duplicated().sum()}")
    print(f"trade_pct range:      [{df['trade_pct'].min():.2f}, {df['trade_pct'].max():.1f}]")

### Interpretation Questions

**Q1:** Which of the 5 bugs would ydata-profiling have caught automatically? Which requires domain knowledge that no automated tool can provide?

| Bug | Auto-detected? | Why or why not? |
|-----|---------------|------------------|
| 1. Negative GDP | | |
| 2. Life expectancy > 120 | | |
| 3. Duplicate country-years | | |
| 4. Inconsistent trade_pct | | |
| 5. GDP unit mismatch | | |

*Your answer here:*

**Q2:** In a production pipeline processing 1M rows daily, would you rely on manual or automated EDA? What's your monitoring strategy?

*Your answer here:*

## Part 4: The eda_utils.py Module (PRODUCTION — 10 min)

Here is the finished module. It saves this lab's checks as three functions you
can import, and it is part of what you submit. Read it against the lab:

- `check_impossible_values` is Part 0's RANGES check, with the limits passed in
  as a dict, because what counts as impossible depends on the column.
- `detect_distribution_shift` is Part 2's `compute_psi`, plus the label.
- `eda_summary` reports shape, missing values, duplicate rows and outlier counts
  in one call, and returns them as a dict.

`df: pd.DataFrame` and `-> dict` are type hints: they document what goes in
and what comes back; Python does not enforce them. `pd.Series | np.ndarray`
means "either of these", and `-> tuple[float, str]` means the function returns
a pair, a number and a text label.

The functions return values and print nothing; the test cell does the
printing. Run the module cell, then the test cell below.

In [ ]:
%%writefile eda_utils.py
"""EDA checks for a data pipeline: impossible values, distribution shift, a summary."""

import numpy as np
import pandas as pd


def check_impossible_values(df: pd.DataFrame, constraints: dict) -> pd.DataFrame:
    """Return a copy of df plus one True/False '{col}_flagged' column per constraint.

    constraints maps a column name to its allowed (min, max).
    """
    result = df.copy()
    for col, limits in constraints.items():
        low, high = limits
        # | means "or", row by row: True where the value is below low or above high
        result[f"{col}_flagged"] = (df[col] < low) | (df[col] > high)
    return result


def detect_distribution_shift(train_col: pd.Series | np.ndarray,
                              inference_col: pd.Series | np.ndarray,
                              n_bins: int = 10) -> tuple[float, str]:
    """Population Stability Index of inference_col against train_col, and its label.

    PSI = sum over bins of (actual - expected) * ln(actual / expected), where
    expected and actual are the shares of each sample that fall in the bin.
    """
    train_arr = np.asarray(train_col, dtype=float)
    inf_arr = np.asarray(inference_col, dtype=float)

    # the bins span BOTH samples, so no value from either falls off the ends
    edges = np.linspace(min(train_arr.min(), inf_arr.min()),
                        max(train_arr.max(), inf_arr.max()),
                        n_bins + 1)
    expected_counts = np.histogram(train_arr, bins=edges)[0]
    actual_counts = np.histogram(inf_arr, bins=edges)[0]

    # adding a small eps keeps an empty bin from producing log(0)
    eps = 1e-4
    expected_pct = expected_counts / len(train_arr) + eps
    actual_pct = actual_counts / len(inf_arr) + eps

    psi = float(np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct)))

    if psi < 0.1:
        label = "stable"
    elif psi < 0.25:
        label = "moderate_shift"
    else:
        label = "significant_shift"
    return psi, label


def eda_summary(df: pd.DataFrame) -> dict:
    """Shape, dtypes, missing values, duplicate rows, describe() and outlier counts, as a dict."""
    # select_dtypes(include="number") keeps only the numeric columns
    numeric = df.select_dtypes(include="number")

    # Tukey's rule: an outlier is more than 1.5 IQRs outside the middle half
    outlier_counts = {}
    for col in numeric.columns:
        q1 = numeric[col].quantile(0.25)
        q3 = numeric[col].quantile(0.75)
        iqr = q3 - q1
        is_outlier = (numeric[col] < q1 - 1.5 * iqr) | (numeric[col] > q3 + 1.5 * iqr)
        outlier_counts[col] = int(is_outlier.sum())

    return {
        "shape": df.shape,
        "dtypes": df.dtypes.to_dict(),
        "missing": df.isna().sum().to_dict(),            # every column, zeros included
        "duplicates": int(df.duplicated().sum()),        # whole-row copies only
        "numeric_summary": numeric.describe().to_dict(),
        "outlier_counts": outlier_counts,                # every numeric column
    }


# runs only as `python eda_utils.py`, not on import
if __name__ == "__main__":
    demo = pd.DataFrame({"gdp": [1200.0, -35.0, 25000.0, 8.4],
                         "trade_pct": [45.0, 0.6, 310.0, 62.0]})
    flagged = check_impossible_values(demo, {"gdp": (0, 1e6), "trade_pct": (0, 200)})
    print(flagged)
    print(detect_distribution_shift(demo["gdp"], demo["gdp"] * 1.3))
    print(eda_summary(demo)["outlier_counts"])

In [ ]:
# TEST — check the module works

import importlib
import eda_utils
importlib.reload(eda_utils)      # picks up a new copy if you re-run the %%writefile cell

print("=== Test 1: check_impossible_values ===")
constraints = {
    "gdp": (0, 100000),
    "life_expectancy": (0, 100),
    "trade_pct": (0, 100),
}
flagged = eda_utils.check_impossible_values(df, constraints)
flag_cols = [f"{col}_flagged" for col in constraints]
# .any(axis=1) is True for a row with at least one flag; .sum() counts those rows
print(f"Flagged {flagged[flag_cols].any(axis=1).sum()} rows with constraint violations.")
for flag in flag_cols:
    print(f"  {flag}: {flagged[flag].sum()} violations")
print()

print("=== Test 2: detect_distribution_shift ===")
psi_val, interp = eda_utils.detect_distribution_shift(
    train["gdp"].values, inference["gdp"].values
)
print(f"GDP PSI = {psi_val:.4f} ({interp})")
assert psi_val > 0.25, f"Expected PSI > 0.25, got {psi_val:.4f}"
print()

print("=== Test 3: eda_summary ===")
report = eda_utils.eda_summary(df_clean)
print(f"Shape: {report['shape']}")
print(f"Duplicates: {report['duplicates']}")

# the report has every column; {col: n for ... if n > 0} keeps only the nonzero ones
missing = {col: n for col, n in report["missing"].items() if n > 0}
outliers = {col: n for col, n in report["outlier_counts"].items() if n > 0}
if missing:
    print(f"Missing values: {missing}")
else:
    print("Missing values: None")
if outliers:
    print(f"Outliers (IQR): {outliers}")
else:
    print("Outliers (IQR): None detected")

print(f"\nReport keys: {list(report.keys())}")

### Verification Checkpoint

Your module tests should confirm:
- `check_impossible_values` flags violations in the corrupted dataset
- `detect_distribution_shift` returns PSI > 0.25 for the shifted GDP column, the same value as Part 2's table
- `eda_summary` runs without error and returns a dict with all expected keys

### Before the dashboard: save the data

This notebook builds its data **in memory**. Nothing is on disk except
`eda_utils.py`, which Part 4 just wrote, so a dashboard opened on its own has
nothing to load, and a chat assistant you paste the task into cannot see the
data either.

The cell below writes the four frames next to the module and zips the lot, so
you can download it in one click and keep working outside this notebook. Unzip
it and put your dashboard in the same folder. Then:

```python
import pandas as pd
import eda_utils

train = pd.read_csv("training.csv")
inference = pd.read_csv("inference.csv")
psi, interp = eda_utils.detect_distribution_shift(train["gdp"], inference["gdp"])
```

gives the same GDP PSI as Part 2's table. Note the shape:
`detect_distribution_shift` takes two **columns**, not two DataFrames, and
returns a (value, label) pair.

In [ ]:
# GIVEN — run as-is: save the data so the dashboard can run outside this notebook
import zipfile

frames = {
    "pipeline_health_dirty.csv": df,          # 260 rows, all five bugs
    "pipeline_health_clean.csv": df_clean,    # after your Part 1 fixes
    "training.csv":              train,       # reference distribution
    "inference.csv":             inference,   # gdp already shifted 1.3x
}
for name, frame in frames.items():
    frame.to_csv(name, index=False)
    print(f"  wrote {name:<28} {frame.shape}")

bundle = "lab3_dashboard_bundle.zip"
with zipfile.ZipFile(bundle, "w") as z:     # "with" closes the zip when the block ends
    for name in frames:
        z.write(name)
    z.write("eda_utils.py")
print(f"\nZipped to {bundle}.")

# In Colab this sends the zip to your browser; anywhere else it is already next to the notebook
try:
    from google.colab import files
    files.download(bundle)
except ImportError:
    pass

---
## AI-Assisted Expansion: Pipeline Health Dashboard

**AI policy: foundations first, expansion second.** You have fixed the five bugs by hand, read the PSI code and its noise problem, and compared your checks with a profiler. From here on, AI may write code for you (the Co-Pilot Rule).

### Your Expansion Task
Build ONE interactive dashboard, inside this notebook (ipywidgets + plotly), that lets the user:
1. Pick a numeric column and see its training vs. inference distributions overlaid
2. Move a PSI threshold slider (0.10 / 0.25) and see which columns are flagged
3. Edit the impossible-value constraints and see which rows `check_impossible_values` flags
4. Compare the corrupted `df` with your `df_clean` side by side

Import `eda_utils`; do not rewrite it.

### P.R.I.M.E. Prompt
Copy and paste this into Claude or ChatGPT:

In [ ]:
# AI EXPANSION — copy the prompt below into Claude, then paste
# its code at the bottom of this cell. Run it and check it.

# [Prep] Act as an expert Python data scientist specializing
# in data-pipeline monitoring and distribution shift.
#
# [Request] I just completed a diagnosis-first lab where I found
# and fixed five planted corruptions in a country panel using
# EDA alone, computed the Population Stability Index between a
# training and an inference split, and used an eda_utils.py
# module with check_impossible_values(df, constraints),
# detect_distribution_shift(train_col, inference_col, n_bins)
# and eda_summary(df). Build ONE interactive dashboard in Colab:
#   1. a column dropdown overlaying train vs inference
#   2. a PSI threshold slider that lists flagged columns
#   3. editable min/max constraints feeding
#      check_impossible_values
#   4. a before/after panel comparing the dirty and clean frames
#
# The data is already in this runtime, uploaded next to
# eda_utils.py (the save cell above writes and zips it):
#   pipeline_health_dirty.csv   260 rows, all five bugs intact
#   pipeline_health_clean.csv   230 rows with the answer key's
#                               fixes (impossible trade dropped)
#   training.csv                138 rows, reference distribution
#   inference.csv                92 rows, gdp already x1.3
# Load them with pandas. Do NOT assume df / df_clean / train /
# inference already exist -- I may be running this in a fresh
# notebook with only those files.
#
# [Iterate] Use ipywidgets, plotly and pandas. `import eda_utils`
# from the same folder and call it as it is -- do not reimplement
# PSI. Note the shape: detect_distribution_shift(train_col,
# inference_col, n_bins=10) takes two COLUMNS, not two DataFrames,
# and returns a (psi, interpretation) pair. Passing frames raises
# "could not convert string to float". No Streamlit, no new .py
# file, no ydata-profiling.
#
# [Mechanism Check] Add inline comments explaining:
#   - how the PSI bins are built: read eda_utils first and say
#     which sample defines the bin edges -- do not assume it is
#     the training set
#   - why the bin proportions get a small epsilon before the log
#   - why 0.10 and 0.25 are conventions, not statistical tests
#   - which of the five corruptions a PSI check would miss
#
# [Evaluate] Explain what the dashboard catches that automated
# profiling did not, and what it still cannot see.

# PASTE AI-GENERATED CODE BELOW:

### Document the interaction

Three short answers. This is what is graded for the expansion, not the code.

1. **What did the AI get wrong or leave out?** Name one line you had to change,
   or one claim in its explanation that your own output contradicts.
2. **How did you verify it?** Point to a number the dashboard shows that you
   also computed by hand earlier in this lab, and say whether they match.
3. **What would you ask differently next time?**

*Your answers here:*

1.
2.
3.

---
## Digital Portfolio: your README

### Generate your README
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own output before you paste it.

In [ ]:
# README prompt — documentation only, no code

# PASTE THIS PROMPT INTO CLAUDE:
#
# "I need help writing a project description for my data science lab.
# **Important Rule:** Do NOT generate any Python code for me.
#
# **What I did in this lab:**
# * Found and fixed [YOUR VALUE] planted data-quality issues in a country
#   panel using EDA alone (negative GDP, life expectancy stored in months, duplicates,
#   mixed percentage units, a GDP unit mismatch)
# * Measured distribution shift with PSI: GDP PSI = [YOUR VALUE] between
#   training and inference
# * Compared manual EDA with ydata-profiling and named what each missed
# * Used an eda_utils.py module with impossible-value, PSI and summary
#   functions
# * Built an interactive pipeline-health dashboard
#
# **Please write a README.md entry including:**
# 1. Project Title: Pipeline Health Check — EDA, Corruption & Distribution Shift
# 2. Objective: one sentence
# 3. Methodology: Bullet points of technical steps
# 4. Key Findings: Summary of results
# Write it plainly, in the first person. No words like 'robust',
# 'comprehensive' or 'leverage'. Use only the numbers I gave you."

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab03-eda-diagnostics`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab03-eda-diagnostics`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab03-eda-diagnostics`, branch `main`,
   commit message `Lab 03 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. `eda_utils.py` is **not** pushed by step 2 — that step sends the notebook only,
   and the file lives in Colab's temporary storage, which is wiped when the
   session ends. Download it first: in the Colab **Files** panel (folder icon,
   left), right-click `eda_utils.py` → **Download**. Then on the repo page use
   **Add file → Upload files**, drag it in, and **Commit changes**. It is part
   of what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.

---

**End of Lab 3.** You found five planted bugs with EDA alone, measured drift
with PSI and saw how noisy it is at a hundred rows, and compared your checks
with an automated profiler. The profiler checks the data against itself; only
you can check it against the world.